# 04 — Barrido multidominio: ¿el efecto es biológico o topológico?

**Pregunta que se probó:** Si la anti-centralidad grado↔V aparece con la misma
fuerza en redes NO biológicas (sociales, financieras) que en conectomas y
redes metabólicas, el efecto es topológico, no una propiedad especial de
sistemas vivos.

**Resultado:** SOBREVIVIÓ como patrón general (con matices resueltos en
notebooks posteriores). El efecto aparece con fuerza comparable en dominios
muy distintos, consistente con que sea una propiedad de la estructura del
grafo bajo el Laplaciano combinatorio, no del contenido biológico.

Ver detalle completo en `paper/SPG_final_v9.docx`, Sección 5.2.


## Fuentes de datos

- **Human brains (118 conectomas individuales, atlas AAL-116)**
  https://networks.skewed.de/net/human_brains
- **C. elegans metabolic**
  https://networks.skewed.de/net/celegans_metabolic
- **Food web (Little Rock)**
  https://networks.skewed.de/net/foodweb_little_rock
- **Bitcoin alpha**
  https://networks.skewed.de/net/bitcoin_alpha
- **Jazz collaboration network**
  https://networks.skewed.de/net/jazz_collab


> **Nota:** de cada dataset se usó únicamente el archivo de tipo `edges` (lista de aristas). No se usaron archivos de nodos ni de metadatos adicionales de Netzschleuder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/HumanBrain'
print("¿existe la carpeta?:", os.path.isdir(CARPETA))
if os.path.isdir(CARPETA):
    print("archivos que veo:")
    for f in sorted(os.listdir(CARPETA)):
        print("  ", f)

¿existe la carpeta?: True
archivos que veo:
   edges (1) (1).csv
   edges (1).csv
   edges (10) (1).csv
   edges (10).csv
   edges (11) (1).csv
   edges (11).csv
   edges (12).csv
   edges (13) (1).csv
   edges (13).csv
   edges (14) (1).csv
   edges (14).csv
   edges (15).csv
   edges (16) (1).csv
   edges (16).csv
   edges (17) (1).csv
   edges (17).csv
   edges (18).csv
   edges (19) (1).csv
   edges (19).csv
   edges (2) (1).csv
   edges (2).csv
   edges (20) (1).csv
   edges (20).csv
   edges (21).csv
   edges (22) (1).csv
   edges (22).csv
   edges (23) (1).csv
   edges (23).csv
   edges (24) (1).csv
   edges (24).csv
   edges (25).csv
   edges (26) (1).csv
   edges (26).csv
   edges (27) (1).csv
   edges (27).csv
   edges (28).csv
   edges (29).csv
   edges (3) (1).csv
   edges (3).csv
   edges (30) (1).csv
   edges (30).csv
   edges (31) (1).csv
   edges (31).csv
   edges (32) (1).csv
   edges (32).csv
   edges (33).csv
   edges (34).csv
   edges (35).csv
   edges (36).csv
   e

In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N; self.lambda2 = ev[k0]

def medir(archivo, nombre):
    df = pd.read_csv(archivo, comment='#', header=None)
    # tomar solo las 2 primeras columnas (source, target) sin importar cuántas haya
    df = df.iloc[:, :2]; df.columns = ['s','t']
    G = nx.Graph(); G.add_edges_from(df[['s','t']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    N = s.N; deg = s.degree
    rv  = spearmanr(deg, s.V)[0]
    rtt = spearmanr(deg, s.tau_tilde)[0]
    dens = A.sum()/(N*(N-1)); cv = deg.std()/deg.mean()
    fiable = "SI" if np.sign(rv)==np.sign(rtt) else "NO-descartar"
    print(f"{nombre:24s} {N:5d} {dens:6.3f} {cv:5.2f} {rv:+7.3f} {rtt:+7.3f} {fiable:>12s}")
    return dict(red=nombre, N=N, dens=dens, cv=cv, sp_V=rv, sp_tt=rtt, fiable=fiable)

print("cargado OK — ahora corre el bloque de medir")


cargado OK — ahora corre el bloque de medir


In [ ]:
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/HumanBrain'  # ajusta si tu ruta es otra
import os

# primero confirmamos QUÉ archivos ve y la ruta correcta
print("archivos .csv en la carpeta:")
for f in sorted(os.listdir(CARPETA)):
    if f.endswith('.csv'):
        print("  ", f)
print("="*60)

resultados = []
print(f"{'red':24s} {'N':>5s} {'dens':>6s} {'CV':>5s} {'Sp(V)':>7s} {'Sp(tt)':>7s} {'fiable':>12s}")
print("-"*72)
for f in sorted(os.listdir(CARPETA)):
    if not f.endswith('.csv'): continue
    try:
        resultados.append(medir(os.path.join(CARPETA, f), f[:-4]))
    except Exception as e:
        print(f"{f[:-4]:24s} ERROR: {str(e)[:40]}")


archivos .csv en la carpeta:
   edges (1) (1).csv
   edges (1).csv
   edges (10) (1).csv
   edges (10).csv
   edges (11) (1).csv
   edges (11).csv
   edges (12).csv
   edges (13) (1).csv
   edges (13).csv
   edges (14) (1).csv
   edges (14).csv
   edges (15).csv
   edges (16) (1).csv
   edges (16).csv
   edges (17) (1).csv
   edges (17).csv
   edges (18).csv
   edges (19) (1).csv
   edges (19).csv
   edges (2) (1).csv
   edges (2).csv
   edges (20) (1).csv
   edges (20).csv
   edges (21).csv
   edges (22) (1).csv
   edges (22).csv
   edges (23) (1).csv
   edges (23).csv
   edges (24) (1).csv
   edges (24).csv
   edges (25).csv
   edges (26) (1).csv
   edges (26).csv
   edges (27) (1).csv
   edges (27).csv
   edges (28).csv
   edges (29).csv
   edges (3) (1).csv
   edges (3).csv
   edges (30) (1).csv
   edges (30).csv
   edges (31) (1).csv
   edges (31).csv
   edges (32) (1).csv
   edges (32).csv
   edges (33).csv
   edges (34).csv
   edges (35).csv
   edges (36).csv
   edges (37) (1).c

In [ ]:
import pandas as pd, numpy as np, networkx as nx, os

CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/HumanBrain'

# 1. ver qué archivos hay
print("archivos:")
for f in os.listdir(CARPETA):
    print("  ", f)
print("="*60)

# 2. tomar el/los csv y examinarlos crudos
for f in sorted(os.listdir(CARPETA)):
    if not f.endswith('.csv'): continue
    ruta = os.path.join(CARPETA, f)
    print(f"\n### {f} ###")
    print("--- primeras líneas crudas ---")
    print(open(ruta).read()[:400])
    df = pd.read_csv(ruta, comment='#', header=None)
    print(f"--- pandas: {df.shape[1]} columnas, {len(df)} filas ---")
    print(df.head(3))
    # distribución de grados
    df2 = df.iloc[:,:2]; df2.columns=['s','t']
    G = nx.Graph(); G.add_edges_from(df2.values)
    G.remove_edges_from(nx.selfloop_edges(G))
    if G.number_of_nodes()>0:
        grados = [d for _,d in G.degree()]
        print(f"--- grados: min={min(grados)} max={max(grados)} distintos={len(set(grados))} ---")


Se truncaron las últimas líneas 5000 del resultado de transmisión.
0,149,,374
0,157,,101
0,158,,366
1,96,,569
1,32,,657
1,199,,4
1,15,,55
1,81,,338
1,116,,394
1,149,,396
1,55,,67
2,64,,163
2,162,,5
2,131,,126
2,5,,386
2,104,,151
2,169,,290
2,141,,97
2,173,,1522
2,80,,555
2,45,,32
2,53,,33
2,88,,568
2,57,,262
2,27,,17
2,25,,5
3,128,,2531
3,100,,844
3,66,,1229
3,67,,171
3,36,,173
3
--- pandas: 4 columnas, 1373 filas ---
   0    1   2    3
0  0  130 NaN  306
1  0   69 NaN  297
2  0   15 NaN  213
--- grados: min=3 max=43 distintos=30 ---

### edges (14) (1).csv ###
--- primeras líneas crudas ---
# source, target, _graphml_edge_id, weight
0,96,,14
0,130,,330
0,15,,196
0,149,,528
0,157,,244
0,158,,54
1,96,,66
1,32,,249
1,15,,2
1,81,,102
1,116,,290
1,149,,143
1,55,,113
1,155,,13
2,131,,23
2,5,,2
2,101,,931
2,104,,580
2,169,,647
2,171,,6
2,141,,108
2,173,,355
2,80,,725
2,45,,28
2,53,,44
2,88,,840
2,57,,180
2,137,,4
2,84,,363
3,128,,1552
3,129,,517
3,136,,66
3,10,,1487
3,172,,209
3,16,,200
3,1


In [ ]:
import numpy as np
sp_tt = [r['sp_tt'] for r in resultados]
sp_V  = [r['sp_V']  for r in resultados]
cv    = [r['cv']    for r in resultados]

print("=== SÍNTESIS: anti-centralidad en conectomas humanos BNU1 ===")
print(f"n = {len(sp_tt)} conectomas individuales")
print(f"Spearman(k,τ̃):  media = {np.mean(sp_tt):+.3f}  ±{np.std(sp_tt):.3f}  "
      f"[min {np.min(sp_tt):+.3f}, max {np.max(sp_tt):+.3f}]")
print(f"Spearman(k,V):   media = {np.mean(sp_V):+.3f}  ±{np.std(sp_V):.3f}")
print(f"CV(grado):       media = {np.mean(cv):.2f}  ±{np.std(cv):.2f}")
print(f"% negativos (τ̃): {100*np.mean(np.array(sp_tt)<0):.1f}%")
print(f"% confiables (V y τ̃ mismo signo): {100*np.mean(np.sign(sp_V)==np.sign(sp_tt)):.1f}%")


=== SÍNTESIS: anti-centralidad en conectomas humanos BNU1 ===
n = 118 conectomas individuales
Spearman(k,τ̃):  media = -0.818  ±0.089  [min -0.939, max -0.317]
Spearman(k,V):   media = -0.994  ±0.003
CV(grado):       media = 0.52  ±0.04
% negativos (τ̃): 100.0%
% confiables (V y τ̃ mismo signo): 100.0%


In [ ]:
import networkx as nx, numpy as np
from scipy.stats import spearmanr
import networkx.algorithms.community as nxcom

def prueba_tau_vs_V(G, nombre):
    s = SPG(nx.to_numpy_array(G))
    nodes = list(G.nodes()); idx = {n:i for i,n in enumerate(nodes)}
    V, tt = s.V, s.tau_tilde
    print(f"\n=== {nombre} (N={len(nodes)}) ===")

    # detectar comunidades (Louvain)
    try:
        comms = nxcom.louvain_communities(G, seed=42)
        comm_of = {n:i for i,c in enumerate(comms) for n in c}
    except Exception as e:
        print("  no se pudo comunidad:", e); return

    # TAREA 1: participation coefficient (rol de conector entre módulos)
    #   nodo conector = conecta a muchas comunidades distintas
    part = []
    for n in nodes:
        degs = {}
        for nb in G.neighbors(n):
            c = comm_of[nb]; degs[c] = degs.get(c,0)+1
        k = G.degree(n)
        P = 1 - sum((d/k)**2 for d in degs.values()) if k>0 else 0
        part.append(P)
    part = np.array(part)

    # TAREA 2: within-module degree (hub de módulo)
    wmd = []
    for n in nodes:
        c = comm_of[n]
        same = [m for m in comms[c]]
        sub = G.subgraph(same)
        wmd.append(sub.degree(n) if n in sub else 0)
    wmd = np.array(wmd, float)

    # TAREA 3: es nodo frontera (tiene vecinos en otra comunidad)?
    frontera = np.array([1 if any(comm_of[nb]!=comm_of[n] for nb in G.neighbors(n)) else 0
                         for n in nodes], float)

    for tarea, y in [('participation', part), ('within_mod_deg', wmd), ('frontera', frontera)]:
        rV  = abs(spearmanr(V, y)[0])
        rtt = abs(spearmanr(tt, y)[0])
        gana = "  <-- TAU GANA" if rtt - rV > 0.15 else ""
        print(f"  {tarea:16s}  |V|={rV:.3f}  |tau|={rtt:.3f}{gana}")

# corre en 3-4 redes CON estructura de comunidades clara y lambda_max/lambda2 BAJO
# (importante: solo redes donde tau_tilde es válido)
# ejemplos: jazz_collab, celegans_metabolic, una budapest _1m
prueba_tau_vs_V(G, "tu_red")



=== tu_red (N=116) ===
  participation     |V|=0.736  |tau|=0.628
  within_mod_deg    |V|=0.828  |tau|=0.760
  frontera          |V|=0.555  |tau|=0.563


In [ ]:
import networkx as nx, numpy as np
from scipy.stats import spearmanr, pearsonr

s = SPG(nx.to_numpy_array(G))
V = s.V
nodes = list(G.nodes())

# L+ diagonal directa (pseudoinversa)
L = nx.laplacian_matrix(G).toarray().astype(float)
Lplus = np.linalg.pinv(L)
Lplus_diag = np.diag(Lplus)

# current-flow closeness de networkx
cfc = np.array([nx.current_flow_closeness_centrality(G)[n] for n in nodes])

print("¿Tu V es EXACTAMENTE L+_ii?")
print(f"  Pearson(V, L+_ii)  = {pearsonr(V, Lplus_diag)[0]:.10f}")
print(f"  máx diferencia abs = {np.max(np.abs(V - Lplus_diag)):.2e}")
print(f"  ¿idénticas? {np.allclose(V, Lplus_diag)}")
print()
print("¿Tu V es EXACTAMENTE current-flow closeness?")
print(f"  Pearson(V, CFC)    = {pearsonr(V, cfc)[0]:.6f}")
print(f"  Spearman(V, CFC)   = {spearmanr(V, cfc)[0]:.6f}")
print(f"  ¿idénticas? {np.allclose(V, cfc)}")
print(f"  ¿idénticas tras escalar? {np.allclose(V/V.mean(), cfc/cfc.mean())}")


¿Tu V es EXACTAMENTE L+_ii?
  Pearson(V, L+_ii)  = 1.0000000000
  máx diferencia abs = 2.44e-15
  ¿idénticas? True

¿Tu V es EXACTAMENTE current-flow closeness?
  Pearson(V, CFC)    = -0.945373
  Spearman(V, CFC)   = -0.999998
  ¿idénticas? False
  ¿idénticas tras escalar? False


In [ ]:
import networkx as nx, numpy as np, pandas as pd, os
from scipy.stats import spearmanr, pearsonr

def comparar_dominio(archivo, nombre, dominio):
    df = pd.read_csv(archivo, comment='#', header=None).iloc[:,:2]
    df.columns=['s','t']
    G = nx.Graph(); G.add_edges_from(df.values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); nodes=list(G.nodes())
    # L+ii directo (la definición canónica, sin pasar por SPG)
    L = np.diag(A.sum(1)) - A
    Lplus_diag = np.diag(np.linalg.pinv(L))
    deg = np.array([G.degree(n) for n in nodes])
    # centralidades clásicas
    cfc = np.array([nx.current_flow_closeness_centrality(G)[n] for n in nodes])
    clo = np.array([nx.closeness_centrality(G)[n] for n in nodes])
    print(f"\n=== {nombre} [{dominio}] N={len(nodes)} ===")
    print(f"  Spearman(grado, L+ii)        = {spearmanr(deg, Lplus_diag)[0]:+.3f}")
    print(f"  Spearman(L+ii, current_flow) = {spearmanr(Lplus_diag, cfc)[0]:+.3f}")
    print(f"  Spearman(L+ii, closeness)    = {spearmanr(Lplus_diag, clo)[0]:+.3f}")
    print(f"  Pearson (L+ii, current_flow) = {pearsonr(Lplus_diag, cfc)[0]:+.3f}")

CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/HumanBrain'
# biológicas y NO biológicas, mezcladas a propósito
casos = [
    ('celegans_metabolic.csv','celegans_metabolic','BIOLÓGICO'),
    ('foodweb_little_rock.csv','foodweb','BIOLÓGICO'),
    ('jazz_collab.csv','jazz','SOCIAL - no bio'),
    ('bitcoin_alpha.csv','bitcoin','FINANCIERO - no bio'),
]
for arch, nom, dom in casos:
    try: comparar_dominio(os.path.join(CARPETA,arch), nom, dom)
    except Exception as e: print(f"{nom}: ERROR {str(e)[:40]}")


celegans_metabolic: ERROR [Errno 2] No such file or directory: '/c
foodweb: ERROR [Errno 2] No such file or directory: '/c
jazz: ERROR [Errno 2] No such file or directory: '/c
bitcoin: ERROR [Errno 2] No such file or directory: '/c
